# Week 10 — Recommendation, Ranking, and Predictive Decision Engine

This notebook runs the full recommendation pipeline from `src/recommendation.py` and produces all visualizations for the Week 10 deliverable.

**Reproducible pipeline:** From the project root, the same logic runs as:
```bash
python src/recommendation.py
```

**Task framing:** This is a **recommendation** task. Given a user's positive interaction history (games they endorsed), we predict which unseen game they would also endorse. Correctness = the held-out game appears in the top-K list.

**Systems compared:**
1. Popularity baseline
2. Content-based baseline (TF-IDF + SVD cosine similarity)
3. Matrix Factorization with SGD (k ∈ {10, 20, 50})
4. Hybrid: α × content + (1−α) × MF

In [ ]:
import os
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Add src to path so we can import recommendation module
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, 'artifacts', 'recommendation')
REPORTS_DIR   = os.path.join(PROJECT_ROOT, 'reports', 'Week10')
os.makedirs(REPORTS_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

## 1. Run pipeline

Running `recommendation.py` executes all four systems and saves artifacts. If artifacts already exist, we load them directly.

In [ ]:
metrics_path = os.path.join(ARTIFACTS_DIR, 'rec_metrics.json')

if not os.path.exists(metrics_path):
    print('Running pipeline...')
    import recommendation
    recommendation.run_pipeline()
else:
    print('Artifacts found — loading existing results.')

with open(metrics_path) as f:
    metrics = json.load(f)

print(f"Users: {metrics['n_users']:,}")
print(f"Games: {metrics['n_games']:,}")
print(f"Qualifying users (hold-out): {metrics['n_qualifying_users']:,}")
print(f"Best MF latent dim k: {metrics['mf_best_k']}")

## 2. Evaluation Summary — HR@K comparison

In [ ]:
ev = metrics['evaluation']
best_k = metrics['mf_best_k']

rows = [
    ('Popularity baseline',  ev['popularity_baseline']),
    ('Content-based',        ev['content_based_baseline']),
    (f'MF k=10',             ev['matrix_factorization'].get('k=10', {})),
    (f'MF k=20',             ev['matrix_factorization'].get('k=20', {})),
    (f'MF k=50',             ev['matrix_factorization'].get('k=50', {})),
]

# Best hybrid
best_alpha, best_hres = max(ev['hybrid'].items(), key=lambda x: x[1].get('HR@10', 0))
rows.append((f'Hybrid ({best_alpha})', best_hres))

summary = pd.DataFrame(
    [{'System': name, 'HR@5': r.get('HR@5', 0), 'HR@10': r.get('HR@10', 0), 'HR@20': r.get('HR@20', 0)}
     for name, r in rows]
).set_index('System')

print(summary.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(summary))
w = 0.25
for i, col in enumerate(['HR@5', 'HR@10', 'HR@20']):
    ax.bar(x + i * w, summary[col], width=w, label=col)
ax.set_xticks(x + w)
ax.set_xticklabels(summary.index, rotation=25, ha='right')
ax.set_ylabel('Hit-Rate')
ax.set_title('HR@K Comparison: Popularity vs Content vs MF vs Hybrid')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig01_hr_comparison.png'))
plt.show()

## 3. MF Training — Loss Curve

In [ ]:
loss_history = metrics['mf_loss_history']

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(loss_history) + 1), loss_history, marker='o', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Regularized training loss')
ax.set_title(f'Matrix Factorization: SGD loss curve (k={best_k})')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig02_mf_loss_curve.png'))
plt.show()

print(f'Loss epoch 1 : {loss_history[0]:.4f}')
print(f'Loss epoch {len(loss_history)}: {loss_history[-1]:.4f}')
print(f'Relative decrease: {(loss_history[0] - loss_history[-1]) / loss_history[0]:.1%}')

## 4. MF Latent Dimension Sweep

In [ ]:
mf_sweep = ev['matrix_factorization']
k_labels = [int(k.split('=')[1]) for k in mf_sweep]
hr10_vals = [mf_sweep[k]['HR@10'] for k in mf_sweep]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_labels, hr10_vals, marker='o', linewidth=2)
ax.set_xlabel('Latent dimension k')
ax.set_ylabel('HR@10')
ax.set_title('MF: HR@10 vs latent dimension k')
ax.set_xticks(k_labels)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig03_mf_k_sweep.png'))
plt.show()

## 5. Hybrid Alpha Sweep

In [ ]:
hybrid = ev['hybrid']
alphas = [float(a.split('=')[1]) for a in hybrid]
hr10_hybrid = [hybrid[a]['HR@10'] for a in hybrid]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, hr10_hybrid, marker='s', linewidth=2, color='darkorange')
ax.axhline(ev['content_based_baseline']['HR@10'], ls='--', color='steelblue', label='Content-only')
ax.axhline(ev['matrix_factorization'][f'k={best_k}']['HR@10'], ls='--', color='green', label=f'MF k={best_k}')
ax.set_xlabel('α (weight on content score)')
ax.set_ylabel('HR@10')
ax.set_title('Hybrid: HR@10 vs α (content weight)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig04_hybrid_alpha_sweep.png'))
plt.show()

print(f'Best alpha: {best_alpha}  HR@10: {best_hres["HR@10"]:.4f}')

## 6. Error Analysis

In [ ]:
ea = metrics['error_analysis']

print('=== HIT EXAMPLES (MF recommended the held-out game) ===')
for ex in ea['hit_examples']:
    print(f"  User: ...{ex['user'][-6:]}")
    print(f"    Train games : {ex['train_games']}")
    print(f"    Held-out    : {ex['held_out_game']}")
    print(f"    Top-3 recs  : {ex['top3_recommended']}")
    print()

print('=== MISS EXAMPLES (MF failed to recommend the held-out game) ===')
for ex in ea['miss_examples']:
    print(f"  User: ...{ex['user'][-6:]}")
    print(f"    Train games : {ex['train_games']}")
    print(f"    Held-out    : {ex['held_out_game']}")
    print(f"    Top-3 recs  : {ex['top3_recommended']}")
    print()

## 7. Interpretation Notes

**On the popularity baseline:** A popularity-based recommender ranks the same top games for every user regardless of history. Its HR@10 establishes the floor — any personalized system should beat it, otherwise added complexity is unjustified.

**On content-based:** Content similarity captures game identity (genre, tags, description). It handles new items well but tends to overspecialize toward the user's known preferences and misses serendipitous but relevant titles.

**On MF:** The latent factor model learns co-occurrence structure from collective behavior. It can surface games that are not obviously similar in tag space but are endorsed by users with similar taste profiles. The regularization term (λ=0.01) controls factor magnitude to prevent overfitting on the sparse training set.

**On the hybrid:** The alpha parameter controls the balance between content evidence (higher α) and behavioral evidence (lower α). The optimal α is an empirical question — neither extreme (pure content or pure MF) is guaranteed to dominate.

**Limitations:**
- V1 subset covers only 500 games (the full universe has ~80k). Results may not generalize to the full catalog.
- The interaction matrix is very sparse (~99.8%). MF on sparse data is sensitive to initialization and regularization.
- Hold-out is random (not temporal). In production, a time-based split would be more realistic.
- Users with only 1 positive interaction are excluded from evaluation, introducing selection bias toward active users.